# Store Sales Machine Learning
## Machine Learning & Prediction

* **Goal:** Develop a machine learning model to predict sales for individual stores and product families based on historical sales data, promotions, calendar features, and other relevant variables.

In [1]:
# Import libraies
from pathlib import Path
import pandas as pd

data_path = Path('../data/train_ml.csv')
train = pd.read_csv(data_path, parse_dates=['date'])

train.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico,year,month,day,day_of_week,is_weekend,quarter,is_special_date,day_of_year
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,1746.0,93.14,2013,1,1,1,0,1,1,1
1,1194,2013-01-01,42,CELEBRATION,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1
2,1193,2013-01-01,42,BREAD/BAKERY,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1
3,1192,2013-01-01,42,BOOKS,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1
4,1191,2013-01-01,42,BEVERAGES,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1


In [2]:
# Inspection
print(f'Rows: {train.shape[0]} -- Cols: {train.shape[1]}')

print(train.columns.tolist())

print(train[['date', 'store_nbr', 'family', 'sales']].head())

Rows: 3000888 -- Cols: 20
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'transactions', 'dcoilwtico', 'year', 'month', 'day', 'day_of_week', 'is_weekend', 'quarter', 'is_special_date', 'day_of_year']
        date  store_nbr        family  sales
0 2013-01-01          1    AUTOMOTIVE    0.0
1 2013-01-01         42   CELEBRATION    0.0
2 2013-01-01         42  BREAD/BAKERY    0.0
3 2013-01-01         42         BOOKS    0.0
4 2013-01-01         42     BEVERAGES    0.0


In [3]:
train['date'].min(), train['date'].max()

(Timestamp('2013-01-01 00:00:00'), Timestamp('2017-08-15 00:00:00'))

In [4]:
# Sort data chronologically
train = train.sort_values('date').reset_index(drop=True)

validation_start = '2017-08-01'

# Split data by date
train_data = train[train['date'] < validation_start].copy()
valid_data = train[train['date'] >= validation_start].copy()

# Check split
print('Trainings period:')
print(f' {train_data['date'].min()} to {train_data['date'].max()}')
print('Validation period:')
print(f' {valid_data['date'].min()} to {valid_data['date'].max()}')

Trainings period:
 2013-01-01 00:00:00 to 2017-07-31 00:00:00
Validation period:
 2017-08-01 00:00:00 to 2017-08-15 00:00:00


In [5]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Calculate average sales for each store and product family
sales_mean = (
    train_data
    .groupby(['store_nbr', 'family'])['sales']
    .mean()
)

# Create predition for vald.perido
valid_prediction = valid_data.set_index(
    ['store_nbr', 'family']
).index.map(sales_mean)

# Replace missing pred. with overall training average
valid_prediction = np.where(
    pd.isna(valid_prediction), 
    train_data['sales'].mean(), 
    valid_prediction
)

# Evaluate baseline
mae = mean_absolute_error(valid_data['sales'], valid_prediction)
rmse = np.sqrt(mean_squared_error(valid_data['sales'], valid_prediction))

print(f'Baseline MAE {mae:.2f}')
print(f'Baseline RMSE: {rmse:.2f}')


Baseline MAE 143.64
Baseline RMSE: 506.74


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

In [7]:
# Define target
target = 'sales'

# Define features
categorial_features = ['family', 'city', 'state', 'type']
numeric_features = [
    'store_nbr',
    'cluster',
    'onpromotion',
    'dcoilwtico', 
    'year',
    'month',
    'day',
    'day_of_week',
    'is_weekend', 
    'quarter', 
    'is_special_date', 
    'day_of_year'
]

features = categorial_features + numeric_features

X_train = train_data[features]
y_train = train_data[target]

X_valid = valid_data[features]
y_valid = valid_data[target]

print('Training features:', X_train.shape)
print('Validation features:', X_valid.shape)

Training features: (2974158, 16)
Validation features: (26730, 16)


In [8]:
# Convert categorial feat into numeric values
preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorial', 
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            categorial_features
        )
    ],
    remainder='passthrough'
)

# Define machine learning model
model = make_pipeline(
    preprocessor, 
    HistGradientBoostingRegressor(
        max_iter=100, 
        learning_rate=0.1, 
        max_leaf_nodes=31, 
        random_state=42
    )
)

print('Ready')

Ready


In [9]:
# Create sample of df
sample_size = min(300_000, len(X_train))

X_train_sample = X_train.sample(
    n=sample_size, 
    random_state=42
)

y_train_sample = y_train.loc[X_train_sample.index]

print('Sample Size:', len(X_train_sample))

# Train model
model.fit(X_train_sample, y_train_sample)


Sample Size: 300000


,steps,"[('columntransformer', ...), ('histgradientboostingregressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorial', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [10]:
# Prediction sales
prediction = model.predict(X_valid)

# Evaluate model
mae = mean_absolute_error(y_valid, prediction)
rmse = np.sqrt(mean_squared_error(y_valid, prediction))

print(f'Baseline MAE: 143.64')
print(f'Model MAE: {mae:.2f}')
print(f'Baseline RMSE: 506.74')
print(f'Model RMSE: {rmse:.2f}')

Baseline MAE: 143.64
Model MAE: 145.27
Baseline RMSE: 506.74
Model RMSE: 347.94


In [11]:
# compare actual and predicted sales
results = valid_data[['date', 'store_nbr', 'family', 'sales']].copy()

results['prediction'] = prediction
results['error'] = results['sales'] - results['prediction']
results['absolute_error'] = results['error'].abs()

print('Mean actual sales:', results['sales'].mean())
print('Mean predicted sales:', results['prediction'].mean())
print('Negative prediction:', (results['prediction'] <0).sum())

print('Largest prediction errors:')
display(
    results.sort_values('absolute_error', ascending=False).head(10)
)

Mean actual sales: 465.144891162645
Mean predicted sales: 492.75904559736625
Negative prediction: 3333
Largest prediction errors:


,date,store_nbr,family,sales,prediction,error,absolute_error
2982642,2017-08-05,47,BEVERAGES,14469.000,8184.388256,6284.611744,6284.611744
2984688,2017-08-06,54,BEVERAGES,8552.000,2502.737534,6049.262466,6049.262466
2994893,2017-08-12,40,GROCERY I,10739.000,4989.661244,5749.338756,5749.338756
2980629,2017-08-04,40,BEVERAGES,8272.000,2652.332404,5619.667596,5619.667596
2996214,2017-08-13,28,GROCERY I,11002.664,5675.684742,5326.979258,5326.979258
2994432,2017-08-12,28,GROCERY I,9841.523,4594.782078,5246.740922,5246.740922
2977002,2017-08-02,39,CLEANING,6263.000,1257.385644,5005.614356,5005.614356
2974887,2017-08-01,3,BEVERAGES,8192.000,3306.956271,4885.043729,4885.043729
2996797,2017-08-13,44,GROCERY I,9811.000,14691.143537,-4880.143537,4880.143537
2995026,2017-08-12,44,GROCERY I,9391.000,14138.097977,-4747.097977,4747.097977


In [12]:
# Transform target to reduce influence of large values
y_train_log = np.log1p(y_train_sample)

# Train model with transformed target
model.fit(X_train_sample, y_train_log)

# Predict and convert back to original
prediction_log = model.predict(X_valid)
prediction_log = np.maximum(np.expm1(prediction_log), 0)

mae_log = mean_absolute_error(y_valid, prediction_log)
rmse_log = np.sqrt(mean_squared_error(y_valid, prediction_log))

print(f'Baseline MAE: 143.64')
print(f'Previous model MAE: 145.27')
print(f'Log model MAE: {mae_log:.2f}')

print(f'Baseline RMSE: 506.74')
print(f'Previous model RMSE: 347.94')
print(f'Log model RMSE: {rmse_log:.2f}')


Baseline MAE: 143.64
Previous model MAE: 145.27
Log model MAE: 146.06
Baseline RMSE: 506.74
Previous model RMSE: 347.94
Log model RMSE: 501.06
